# 第 4 章 過学習・未学習と正則化

多項式回帰で次数を変えながら、訓練誤差とテスト誤差の差（汎化ギャップ）を観察し、L1 / L2 正則化の効果を確かめます。

対応する記事: [第 4 章 過学習・未学習と正則化（F# 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/fsharp/ch04.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch04Regularization.fs"

open GrokkingMl.Ch04Regularization

## データセット

`y = 2x + 3` に小さなノイズを乗せた 10 点です。**真の関係は 1 次関数** なので、高次のモデルは過学習するはずです。

In [2]:
let features = [ -1.5; -1.2; -0.9; -0.6; -0.3; 0.0; 0.3; 0.6; 0.9; 1.2 ]
let labels = [ 0.08; 0.32; 1.07; 1.63; 2.54; 3.11; 3.84; 3.95; 4.75; 5.12 ]

let split = trainTestSplit 0.3 0 features labels
printfn "訓練 %A" split.TrainFeatures
printfn "テスト %A" split.TestFeatures

訓練 

[-0.6; 0.0; -1.5; -0.9; -1.2; 0.3; 0.9]

テスト 

[-0.3; 1.2; 0.6]

## 次数を上げるとどうなるか

**訓練誤差は下がるのに、汎化ギャップ（テスト誤差 − 訓練誤差）は広がります。** これが過学習です。

In [3]:
printfn "%4s %10s %12s %10s" "次数" "訓練 RMSE" "テスト RMSE" "ギャップ"

for degree in [ 1; 3; 5 ] do
    let m = train degree NoRegularization 0.0 split.TrainFeatures split.TrainLabels
    let trainError = modelRmse m split.TrainFeatures split.TrainLabels
    let testError = modelRmse m split.TestFeatures split.TestLabels
    printfn "%4d %10.4f %12.4f %10.4f" degree trainError testError (testError - trainError)

  次数

   訓練 RMSE

    テスト RMSE

      ギャップ

   1

...

    0.2408

   5

    0.0537

      0.3370

    0.2833

## 正則化を掛ける

5 次モデルに λ = 0.01 の正則化を掛けます。**訓練誤差は悪化しますが、テスト誤差は改善します。** 訓練データへの当てはまりをわざと諦めて、未知データへの当てはまりを買っています。

In [4]:
printfn "%-8s %8s %8s %10s" "正則化" "訓練" "テスト" "重みの合計"

for (kind, strength) in [ NoRegularization, 0.0; L1, 0.01; L2, 0.01 ] do
    let m = train 5 kind strength split.TrainFeatures split.TrainLabels
    printfn "%-8A %8.4f %8.4f %10.3f" kind (modelRmse m split.TrainFeatures split.TrainLabels)
        (modelRmse m split.TestFeatures split.TestLabels) (weightMagnitude m)

正則化     

      訓練

     テスト

     重みの合計

NoRegularization

...

     2.467

L2

  0.1233

  0.1498

     2.920

## L1 は重みを 0 にする

学習された 5 つの重みを並べます。**L1 は不要な次数の重みをほぼ 0 まで押し下げます**（スパース性）。L2 は全体をなだらかに縮めるだけで、0 にはしません。

In [5]:
for (kind, strength) in [ NoRegularization, 0.0; L1, 0.01; L2, 0.01 ] do
    let m = train 5 kind strength split.TrainFeatures split.TrainLabels
    let weights = m.Weights |> List.map (sprintf "%8.4f") |> String.concat "  "
    printfn "%-6A %s" kind weights

NoRegularization

  2.2361   -0.2056   -0.0166   -0.1449   -0.1896

L1

  2.1521   -0.2203   -0.0011   -0.0001   -0.0938

L2

  1.7412   -0.3502    0.5119    0.1159   -0.2005

## 試してみる

正則化の強さを変えるとどうなるでしょうか。**強すぎるとかえって悪化します。** 直感に反しますが、重みを潰しすぎると境界の位置そのものが崩れるためです。

In [6]:
for strength in [ 0.0; 0.005; 0.01; 0.05; 0.2 ] do
    let m = train 5 L2 strength split.TrainFeatures split.TrainLabels
    printfn "λ = %-6f テスト RMSE %.4f" strength (modelRmse m split.TestFeatures split.TestLabels)

λ = 

0.000000

 テスト RMSE 

0.3370

λ = 

...

0.1381

λ = 

0.200000

 テスト RMSE 

0.4248